# 10.3 SQLite in Python

**Prerequisites:** 10.1 Introduction to SQL, 06 Exception Handling (context managers)  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What SQLite is, and when a serverless database is the right choice
- 🔴 `sqlite3.sqlite_version` — and the attributes removed in 3.14
- Connections, cursors, and the PEP 249 shape
- Creating tables, and the constraints SQLite enforces
- 🔴 **Parameterised queries and SQL injection**
- `executemany`, named placeholders, and the single-tuple trap
- Fetching: `fetchone` / `fetchmany` / `fetchall` / iteration, and `sqlite3.Row`
- **Transactions** — `commit`, `rollback`, and `with conn:`
- Foreign keys, and why they are off by default
- `:memory:` databases for testing

---

## Sqlite
SQLite is embedded relational DB management system.
> It is self-contained, serverless, zero configuration and transactional SQL DB engine.

- [DB Browser for SQLite](https://sqlitebrowser.org/dl/)

## Stage 1

### Import sqlite library

In [ ]:
import sqlite3 as db

### 🔴 `sqlite3.version` was removed in Python 3.14

The original notebook read `db.version` here, described as *"the version of pysqlite, the
binding of the Python language to the SQLite DB"*.

> | Attribute | Status |
> |---|---|
> | `sqlite3.version` | Deprecated in **3.12**, **removed in 3.14** |
> | `sqlite3.version_info` | Deprecated in **3.12**, **removed in 3.14** |
> | `sqlite3.sqlite_version` | ✅ Still current |
> | `sqlite3.sqlite_version_info` | ✅ Still current |

The removed attributes reported the version of the old external `pysqlite` package, which
has been bundled into the standard library for years — so the number was meaningless. What
you actually want is **`sqlite_version`**: the version of the underlying SQLite *library*,
which determines which SQL features are available.

In [ ]:
import sqlite3 as db
import sys

# The Python version - use this, not the removed sqlite3.version
print("Python           :", ".".join(map(str, sys.version_info[:3])))

# ✅ The SQLite library version - this is the one that matters
print("SQLite library   :", db.sqlite_version)
print("as a tuple       :", db.sqlite_version_info)

# 🔴 The removed attributes
for name in ("version", "version_info"):
    if hasattr(db, name):
        print(f"sqlite3.{name:<13}: {getattr(db, name)}  (deprecated - do not use)")
    else:
        print(f"sqlite3.{name:<13}: REMOVED in this Python")

# Why sqlite_version matters: it gates SQL features
print()
for feature, needed in [("RETURNING clause", (3, 35, 0)),
                        ("STRICT tables", (3, 37, 0)),
                        ("RIGHT/FULL OUTER JOIN", (3, 39, 0))]:
    ok = db.sqlite_version_info >= needed
    print(f"  {feature:<24} needs {'.'.join(map(str, needed)):<8} available: {ok}")

`sqlite_version` gives us the version of the SQLite DB library

In [ ]:
db.sqlite_version

### Establish connection to DB
- **connect():** This method creates a connection object that establish connection between Python file and sqlite3 DB. Simply we can say it represents the DB.
- If the passed DB does not exist, then this method will first created it and finally a DB object will be returned.

In [ ]:
conn = db.connect('Masterly.DB')
print("Connection Established")

We can also create temproary DB in RAM but complete DB and stored data will get vanish as soon as the program is turned off.
- Use argument ":memory:" to create a temporary DB
```python3
conn = sqlite3.connect(':memory:')
```

### Close DB Connection
- **close():** This method closes the DB connection.

In [ ]:
conn.close()

- **NOTE:** This doesn't automatically call commit(). If we just close our database connection without calling commit() first, our changes will be lost!

***

## Stage 2

In [ ]:
import sqlite3 as db
conn = db.connect('Masterly.DB')

### Create cursor object
- **cursor():** This is a method of connection object. It creates cursor object which is used to invoke methods that execute SQL statements in DB.

In [ ]:
cur = conn.cursor()
print("Cursor object created")

### Commands to DB
- **execute():** This is a method of cursor object. It is used to executes an SQL statement.
- SQL statement may be parameterized (i,e. placeholders instead of SQL literals).
- The sqlite3 module supports two kinds of placeholders: question marks and named placeholders (named style).

### Create Table

The original notebook stopped here — a heading followed by an empty cell. Everything from
this point on is new.

We will build a small **inventory** database: products, and the orders placed against them.

In [ ]:
import sqlite3
import tempfile
from pathlib import Path

# A scratch directory, so this notebook never leaves a database file in your
# project folder - and never fails because a previous run still holds a lock.
WORKDIR = Path(tempfile.mkdtemp(prefix="sqlite_demo_"))
DB_PATH = WORKDIR / "inventory.db"
print("database:", DB_PATH.name, "in", WORKDIR.name)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("""
CREATE TABLE product (
    id       INTEGER PRIMARY KEY AUTOINCREMENT,
    sku      TEXT    NOT NULL UNIQUE,
    name     TEXT    NOT NULL,
    price    REAL    NOT NULL CHECK (price >= 0),
    in_stock INTEGER NOT NULL DEFAULT 0
);
""")

cur.execute("""
CREATE TABLE customer_order (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id INTEGER NOT NULL REFERENCES product(id),
    quantity   INTEGER NOT NULL CHECK (quantity > 0),
    placed_at  TEXT    NOT NULL DEFAULT (datetime('now'))
);
""")

conn.commit()

cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
print("tables:", [row[0] for row in cur.fetchall()])

print()
for row in cur.execute("PRAGMA table_info(product);"):
    print(f"  {row[1]:<10} {row[2]:<8} notnull={row[3]} default={row[4]} pk={row[5]}")

print("""
Notes:
  AUTOINCREMENT   SQLite's spelling (MySQL: AUTO_INCREMENT, Postgres: SERIAL)
  REFERENCES      declares a foreign key - but see the PRAGMA note below
  datetime('now') a SQL-side default, stored as TEXT in UTC
""")

### 🔴 Parameterised queries, and SQL injection

This is the single most important section in the folder.

There are two ways to get a Python value into a SQL statement. One of them is a security
vulnerability.

```python
# 🔴 NEVER - string formatting builds the SQL from untrusted text
cur.execute(f"SELECT * FROM product WHERE sku = '{sku}'")

# ✅ ALWAYS - the ? is a PLACEHOLDER; the value never becomes SQL
cur.execute("SELECT * FROM product WHERE sku = ?", (sku,))
```

### Why the first one is dangerous

With string formatting, the *value* and the *code* are concatenated into one string, and the
database cannot tell them apart. If the value contains SQL syntax, that syntax **executes**.

Set `sku` to `' OR '1'='1` and the query becomes:

```sql
SELECT * FROM product WHERE sku = '' OR '1'='1'
```

— which returns every row. Set it to `'; DROP TABLE product; --` and it does exactly what it
looks like.

With a placeholder, the driver sends the SQL and the values to SQLite **separately**. The
value is never parsed as SQL, so there is nothing to inject.

> ### The rule
> **Never build SQL with f-strings, `%`, `.format()` or `+`.** Not for values you trust, not
> for "just this once", not for internal tools. Use placeholders every time, and the question
> never arises.

| Driver | Placeholder |
|---|---|
| `sqlite3` | `?` (positional) or `:name` (named) |
| `mysql-connector` / `PyMySQL` | `%s` |
| `psycopg` (PostgreSQL) | `%s` |

⚠️ Note that `%s` is a **placeholder token**, not Python's `%` formatting. It looks identical
and is completely different.

In [ ]:
# ---- Set up some data, safely ----
products = [
    ("KB-01", "Mechanical Keyboard", 1299.50, 12),
    ("MN-27", "27-inch Monitor",    12499.00, 5),
    ("CB-USB", "USB-C Cable",         349.99, 40),
]

cur.executemany(
    "INSERT INTO product (sku, name, price, in_stock) VALUES (?, ?, ?, ?)",
    products,
)
conn.commit()
print("inserted", cur.rowcount, "products")


# ---- 🔴 The vulnerable version ----
def find_product_unsafe(sku: str):
    """NEVER write this."""
    sql = f"SELECT sku, name, price FROM product WHERE sku = '{sku}'"
    print("  SQL sent:", sql)
    return cur.execute(sql).fetchall()


print("\nlegitimate lookup:")
print("  ->", find_product_unsafe("KB-01"))

print("\n🔴 the same function, attacked:")
print("  ->", find_product_unsafe("' OR '1'='1"))
print("  ^ every row returned - the input became part of the query")


# ---- ✅ The safe version ----
def find_product(sku: str):
    return cur.execute(
        "SELECT sku, name, price FROM product WHERE sku = ?", (sku,)
    ).fetchall()


print("\nsafe version, legitimate lookup:")
print("  ->", find_product("KB-01"))

print("\nsafe version, same attack:")
print("  ->", find_product("' OR '1'='1"))
print("  ^ no rows - the value was searched for literally, exactly as intended")

In [ ]:
# ---- The destructive version, demonstrated on a throwaway table ----
cur.execute("CREATE TABLE victim (id INTEGER PRIMARY KEY, secret TEXT);")
cur.execute("INSERT INTO victim (secret) VALUES ('confidential');")
conn.commit()

print("tables before:", [r[0] for r in cur.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])

# An attacker supplies this as a 'search term'
malicious = "x'; DROP TABLE victim; --"

# sqlite3 refuses multiple statements in execute(), which blocks THIS attack.
# executescript() does not - and plenty of other drivers never did.
try:
    cur.executescript(f"SELECT * FROM victim WHERE secret = '{malicious}'")
except sqlite3.Error as exc:
    print("\nerror:", exc)

print("\ntables after:", [r[0] for r in cur.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])
print("^ `victim` is gone")

print("""
sqlite3.execute() only allows ONE statement, which happens to stop the
DROP TABLE form of this attack. Do not take comfort from that:

  - The '1'='1' data-leak attack above works fine through execute()
  - executescript() has no such protection
  - MySQL and PostgreSQL drivers can be configured to allow multi-statement
  - UPDATE and DELETE injections need only one statement

Placeholders are the defence. Nothing else is.
""")

# Put the table back for later cells
cur.execute("CREATE TABLE IF NOT EXISTS victim (id INTEGER PRIMARY KEY, secret TEXT);")
cur.execute("DROP TABLE victim;")
conn.commit()

### Named placeholders

Positional `?` is fine for two or three values. Past that, **named** placeholders are far
easier to read and impossible to mis-order.

```python
cur.execute(
    "INSERT INTO product (sku, name, price) VALUES (:sku, :name, :price)",
    {"sku": "KB-02", "name": "Compact Keyboard", "price": 899.0},
)
```

⚠️ **A single-value tuple needs its trailing comma** — `(sku,)`, not `(sku)`. Without it you
are passing a string, and `sqlite3` will complain that it received the wrong number of
bindings (**2.3**).

In [ ]:
# ---- Named placeholders, with a dict ----
cur.execute(
    """
    INSERT INTO product (sku, name, price, in_stock)
    VALUES (:sku, :name, :price, :stock)
    """,
    {"sku": "MS-05", "name": "Wireless Mouse", "price": 799.00, "stock": 22},
)
conn.commit()

for row in cur.execute("SELECT sku, name, price FROM product ORDER BY id;"):
    print(" ", row)


# ---- ⚠️ The single-element tuple trap ----
sku = "KB-01"
try:
    cur.execute("SELECT name FROM product WHERE sku = ?", (sku))     # no comma!
except (ValueError, sqlite3.ProgrammingError) as exc:
    print("\n(sku) without a comma:", type(exc).__name__, "-", exc)

print("(sku,) with a comma     :", cur.execute(
    "SELECT name FROM product WHERE sku = ?", (sku,)).fetchone())


# ---- executemany for bulk inserts ----
more = [
    {"sku": "HD-01", "name": "External HDD", "price": 4599.0, "stock": 8},
    {"sku": "SP-02", "name": "Bluetooth Speaker", "price": 2199.0, "stock": 15},
]
cur.executemany(
    "INSERT INTO product (sku, name, price, in_stock) "
    "VALUES (:sku, :name, :price, :stock)",
    more,
)
conn.commit()
print("\ntotal products:", cur.execute("SELECT COUNT(*) FROM product").fetchone()[0])

### Fetching results

`execute()` returns the cursor. How you get rows out of it is a memory decision.

| Method | Returns | Use when |
|---|---|---|
| `fetchone()` | One row, or `None` | You expect exactly one |
| `fetchmany(n)` | Up to `n` rows | Processing in batches |
| `fetchall()` | **Every** row, as a list | The result set is small |
| Iterating the cursor | One row at a time, lazily | **The default** — constant memory |

⚠️ `fetchall()` on a million-row table loads a million rows into memory. Iterating the cursor
does not — it is the `for line in file` of databases (**8.1**).

### `row_factory` — getting rows you can read

By default a row is a plain **tuple**, so you index it by position: `row[2]`. Six months
later nobody knows what `2` was. Set `conn.row_factory = sqlite3.Row` and you can use
`row["price"]` instead — the same argument as `namedtuple` over tuple in **2.3**.

In [ ]:
# ---- Default: rows are tuples ----
cur.execute("SELECT sku, name, price FROM product LIMIT 2;")
print("plain tuples:")
for row in cur.fetchall():
    print("  ", row, "-> price is row[2] =", row[2])


# ---- sqlite3.Row: index by NAME ----
conn.row_factory = sqlite3.Row
cur = conn.cursor()                       # the factory applies to new cursors

cur.execute("SELECT sku, name, price, in_stock FROM product ORDER BY price DESC;")
rows = cur.fetchall()

print("\nwith sqlite3.Row:")
for row in rows:
    print(f"  {row['sku']:<7} {row['name']:<22} {row['price']:>9,.2f}  stock={row['in_stock']}")

print("\nkeys():", rows[0].keys())
print("still indexable by position:", rows[0][0])
print("convert to a dict:", dict(rows[0]))


# ---- fetchone / fetchmany / iteration ----
print("\nfetchone:", dict(cur.execute(
    "SELECT sku, name FROM product WHERE sku = ?", ("MN-27",)).fetchone()))

print("\nmissing row returns None:", cur.execute(
    "SELECT sku FROM product WHERE sku = ?", ("NOPE",)).fetchone())

print("\nfetchmany(2):")
cur.execute("SELECT sku FROM product ORDER BY id;")
while batch := cur.fetchmany(2):
    print("  batch:", [r["sku"] for r in batch])

print("\niterating the cursor - constant memory, the default choice:")
for row in cur.execute("SELECT sku, price FROM product WHERE price > ? ORDER BY price", (1000,)):
    print(f"  {row['sku']:<7} {row['price']:>9,.2f}")

### Transactions

A **transaction** groups statements so they either **all** take effect or **none** do. That
is what stops a crash halfway through leaving your data half-updated — the database
equivalent of the atomic-write pattern in **8.1**.

| Call | Effect |
|---|---|
| `conn.commit()` | Make every change since the last commit permanent |
| `conn.rollback()` | Discard every change since the last commit |
| `with conn:` | Commit on success, **rollback on exception** |

> ⚠️ **`with conn:` is not `with open(...)`.** It manages the *transaction*, **not the
> connection** — it does **not** close it. You still need `conn.close()`, or
> `contextlib.closing()` (**6.3**).

The classic example is moving stock: decrement one row, insert another. If the second fails,
the first must be undone.

In [ ]:
# ---- Without a transaction: a partial update survives ----
def sell_unsafe(sku: str, quantity: int) -> None:
    cur.execute("UPDATE product SET in_stock = in_stock - ? WHERE sku = ?", (quantity, sku))
    conn.commit()                                     # committed too early!
    cur.execute(
        "INSERT INTO customer_order (product_id, quantity) "
        "VALUES ((SELECT id FROM product WHERE sku = ?), ?)",
        (sku, -1),                                    # CHECK (quantity > 0) will reject this
    )
    conn.commit()


before = cur.execute("SELECT in_stock FROM product WHERE sku = 'MN-27'").fetchone()["in_stock"]
try:
    sell_unsafe("MN-27", 2)
except sqlite3.IntegrityError as exc:
    print("insert failed:", exc)

after = cur.execute("SELECT in_stock FROM product WHERE sku = 'MN-27'").fetchone()["in_stock"]
print(f"stock {before} -> {after}  🔴 decremented, but no order was recorded")

cur.execute("UPDATE product SET in_stock = ? WHERE sku = 'MN-27'", (before,))
conn.commit()


# ---- ✅ With a transaction: all or nothing ----
def sell(sku: str, quantity: int) -> None:
    with conn:                                        # commits, or rolls back on exception
        conn.execute("UPDATE product SET in_stock = in_stock - ? WHERE sku = ?",
                     (quantity, sku))
        conn.execute(
            "INSERT INTO customer_order (product_id, quantity) "
            "VALUES ((SELECT id FROM product WHERE sku = ?), ?)",
            (sku, quantity),
        )


before = cur.execute("SELECT in_stock FROM product WHERE sku = 'MN-27'").fetchone()["in_stock"]

try:
    sell("MN-27", -5)                                 # invalid quantity
except sqlite3.IntegrityError as exc:
    print("\ntransaction failed:", exc)

after = cur.execute("SELECT in_stock FROM product WHERE sku = 'MN-27'").fetchone()["in_stock"]
print(f"stock {before} -> {after}  ✅ rolled back - nothing changed")

sell("MN-27", 2)
final = cur.execute("SELECT in_stock FROM product WHERE sku = 'MN-27'").fetchone()["in_stock"]
orders = cur.execute("SELECT COUNT(*) AS n FROM customer_order").fetchone()["n"]
print(f"\nvalid sale: stock {before} -> {final}, orders recorded: {orders}")

### Foreign keys are OFF by default in SQLite

A trap that surprises everyone. SQLite parses `REFERENCES` and stores it in the schema — but
**does not enforce it** unless you switch it on, per connection:

```python
conn.execute("PRAGMA foreign_keys = ON")
```

This is for backwards compatibility with databases created before SQLite supported foreign
keys. It means an unconfigured connection will happily insert an order pointing at a product
that does not exist.

In [ ]:
# ---- Off by default ----
print("foreign_keys:", cur.execute("PRAGMA foreign_keys").fetchone()[0], "(0 = off)")

with conn:
    conn.execute("INSERT INTO customer_order (product_id, quantity) VALUES (?, ?)", (9999, 1))
print("🔴 inserted an order for product 9999, which does not exist")

orphans = cur.execute("""
    SELECT o.id, o.product_id
    FROM customer_order o
    LEFT JOIN product p ON p.id = o.product_id
    WHERE p.id IS NULL
""").fetchall()
print("orphaned orders:", [dict(r) for r in orphans])

# clean up and switch enforcement on
with conn:
    conn.execute("DELETE FROM customer_order WHERE product_id = 9999")
conn.execute("PRAGMA foreign_keys = ON")
print("\nforeign_keys:", cur.execute("PRAGMA foreign_keys").fetchone()[0], "(1 = on)")

try:
    with conn:
        conn.execute("INSERT INTO customer_order (product_id, quantity) VALUES (?, ?)", (9999, 1))
except sqlite3.IntegrityError as exc:
    print("✅ now rejected:", exc)

### Putting it together

A small data-access layer showing the shape you would actually write: a context manager for
the connection, parameterised queries throughout, `sqlite3.Row` for readable results, and
transactions around writes.

In [ ]:
import sqlite3
from contextlib import contextmanager
from pathlib import Path


@contextmanager
def get_connection(path: Path):
    """Open a configured connection, and always close it (see 6.3)."""
    connection = sqlite3.connect(path)
    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA foreign_keys = ON")
    try:
        yield connection
    finally:
        connection.close()


def low_stock(connection, threshold: int) -> list[dict]:
    rows = connection.execute(
        "SELECT sku, name, in_stock FROM product WHERE in_stock < ? ORDER BY in_stock",
        (threshold,),
    ).fetchall()
    return [dict(r) for r in rows]


def order_summary(connection) -> list[dict]:
    rows = connection.execute("""
        SELECT p.sku,
               p.name,
               COUNT(o.id)                 AS orders,
               COALESCE(SUM(o.quantity),0) AS units,
               ROUND(COALESCE(SUM(o.quantity),0) * p.price, 2) AS revenue
        FROM product p
        LEFT JOIN customer_order o ON o.product_id = p.id
        GROUP BY p.id
        ORDER BY revenue DESC
    """).fetchall()
    return [dict(r) for r in rows]


with get_connection(DB_PATH) as connection:
    print("low stock (< 15):")
    for item in low_stock(connection, 15):
        print(f"  {item['sku']:<7} {item['name']:<22} {item['in_stock']:>3}")

    print()
    print(f"  {'sku':<7} {'product':<22} {'orders':>6} {'units':>6} {'revenue':>10}")
    for item in order_summary(connection):
        print(f"  {item['sku']:<7} {item['name']:<22} {item['orders']:>6} "
              f"{item['units']:>6} {item['revenue']:>10,.2f}")

print("\nconnection closed:", True)

### `:memory:` databases — the testing trick

Passing `":memory:"` gives you a complete SQLite database that lives in RAM and disappears
when the connection closes.

That makes it ideal for **tests**: every test gets a fresh, isolated database in
microseconds, with no files to clean up and no server to run. It is one of the main reasons
SQLite is worth knowing even on projects that use PostgreSQL in production.

In [ ]:
import sqlite3

SCHEMA = """
CREATE TABLE product (
    id    INTEGER PRIMARY KEY,
    sku   TEXT NOT NULL UNIQUE,
    price REAL NOT NULL
);
"""


def make_test_db() -> sqlite3.Connection:
    """A fresh, isolated database - no files, no server, no cleanup."""
    connection = sqlite3.connect(":memory:")
    connection.row_factory = sqlite3.Row
    connection.executescript(SCHEMA)
    return connection


def total_value(connection) -> float:
    row = connection.execute("SELECT COALESCE(SUM(price), 0) AS total FROM product").fetchone()
    return row["total"]


# Each "test" is completely independent
for label, rows in [("empty",   []),
                    ("one row", [("A-1", 10.0)]),
                    ("three",   [("A-1", 10.0), ("B-2", 5.5), ("C-3", 1.25)])]:
    db = make_test_db()
    db.executemany("INSERT INTO product (sku, price) VALUES (?, ?)", rows)
    print(f"  {label:<8} total = {total_value(db):>6.2f}")
    db.close()

# ---- Copying a database with backup() ----
source = make_test_db()
source.execute("INSERT INTO product (sku, price) VALUES (?, ?)", ("Z-9", 99.0))

# 🔴 COMMIT FIRST. An uncommitted INSERT holds a write lock, and backup()
# will block on it indefinitely - the notebook simply hangs with no error.
source.commit()

target = sqlite3.connect(":memory:")
source.backup(target)                      # 3.7+
print()
print("after backup(), target has:",
      [tuple(r) for r in target.execute("SELECT sku, price FROM product")])

source.close()
target.close()

# ---- Tidy up the file-based database from earlier ----
import shutil

conn.close()                       # close BEFORE deleting - Windows locks open files
shutil.rmtree(WORKDIR, ignore_errors=True)
print("scratch directory removed:", not WORKDIR.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Building SQL with f-strings or `%`.** That is SQL injection. Use `?` or `:name` placeholders — every time, without exception.
2. 🔴 **Using `sqlite3.version`.** Removed in Python 3.14. Use `sqlite3.sqlite_version`.
3. **Forgetting `commit()`.** Changes are lost when the connection closes. `close()` does not commit.
4. **Thinking `with conn:` closes the connection.** It manages the *transaction*. You still need `conn.close()`.
5. **Forgetting `PRAGMA foreign_keys = ON`.** SQLite ignores foreign keys by default, per connection.
6. **A single-value tuple without the trailing comma** — `(sku)` is a string, `(sku,)` is a tuple.
7. **`fetchall()` on a large table.** Iterate the cursor instead.
8. **Indexing rows by position.** Set `conn.row_factory = sqlite3.Row` and use names.
9. **Assuming SQLite enforces column types.** They are advisory unless the table is `STRICT` (3.37+) — a TEXT column will accept a number.
10. **Sharing one connection across threads** without care. Pass `check_same_thread=False` only if you understand the locking (**12**).

## Best Practices

- **Always** use placeholders. Treat any f-string next to `execute()` as a bug.
- Set `conn.row_factory = sqlite3.Row` immediately after connecting.
- Turn on `PRAGMA foreign_keys = ON` on every connection.
- Wrap related writes in `with conn:` so they commit or roll back together.
- Close connections with a context manager (`contextlib.closing`, or your own).
- Iterate cursors rather than calling `fetchall()`.
- Keep SQL in named constants or functions, not inline in business logic.
- Use `:memory:` databases in tests — fast, isolated, and nothing to clean up.
- Prototype on SQLite; the PEP 249 shape transfers to MySQL and PostgreSQL unchanged.

## Practice Exercises

Try these before moving on.

1. Write `add_product(sku, name, price)` using named placeholders, and prove an injection attempt is stored as literal text.
2. Demonstrate the `' OR '1'='1` attack against an f-string query, then fix it.
3. Write a transfer function that moves stock between two products, and show it rolls back when the second update fails.
4. Query a table with and without `sqlite3.Row` and compare how the code reads.
5. Insert 10,000 rows with `executemany` in one transaction, then with a commit per row. Time both.
6. Turn foreign keys on and off and show the difference in what SQLite accepts.
7. Write a `get_connection()` context manager that always closes, even on exception.
8. Build an in-memory database in a test, insert three rows, and assert an aggregate.
9. Read the whole product table twice: once with `fetchall()`, once by iterating. Compare peak memory for 100,000 rows.